# 04 · LLM judges and arena ratings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/evals/04_judges_and_arenas.ipynb) &nbsp; 📖 Concept deck: [Arenas and ratings](https://nilesh-salpe.github.io/visual-ai/evals/07-arenas/) &nbsp; ⏱️ Suggested time: **45 min**

**You will build:** a position-swap consistency test for an LLM judge, a Bradley-Terry fit from pairwise votes (as Chatbot Arena does), and an online Elo update, plus a demonstration that Elo depends on vote order while Bradley-Terry does not.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def sigmoid(z):
    return 1 / (1 + np.exp(-z))

rng = np.random.default_rng(0)
TRUE = np.array([1.2, 0.8, 0.5, 0.0, -0.3, -1.0])     # true strengths of 6 models (log-odds scale)
M = len(TRUE)
def simulate_votes(n, seed=0):
    """Random pairs of models; the user prefers i over j with prob sigmoid(TRUE[i] - TRUE[j])."""
    r = np.random.default_rng(seed)
    i = r.integers(0, M, n); j = (i + r.integers(1, M, n)) % M
    i_wins = r.random(n) < sigmoid(TRUE[i] - TRUE[j])
    return np.stack([np.where(i_wins, i, j), np.where(i_wins, j, i)], 1)   # rows: (winner, loser)
VOTES = simulate_votes(20000)

def make_judge(first_bias, seed=0):
    """A noisy judge that adds `first_bias` (in log-odds) to whichever answer is shown first."""
    r = np.random.default_rng(seed)
    def judge(qa, qb):
        return 'A' if r.random() < sigmoid(qa - qb + first_bias) else 'B'
    return judge

## Task 1: Position-swap consistency

MT-Bench checked judges by asking twice, with the two answers swapped. A judge is **consistent** on a pair if it picks the same underlying answer both times. GPT-4 was consistent 65% of the time, GPT-3.5 46% and Claude-v1 24% (default prompt), with most inconsistency favouring the first position.
`judge(qa, qb)` returns 'A' (first shown) or 'B'. For each `(qa, qb)` quality pair, query `judge(qa, qb)` and `judge(qb, qa)`. Return `(consistency, first_bias_rate)`, where the latter is the fraction of pairs where the judge picked the **first** answer both times.

In [ ]:
def swap_consistency(judge, pairs):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_swap_consistency(f):
    pairs = [(rng.normal(), rng.normal()) for _ in range(3000)]
    c0, b0 = f(make_judge(0.0, 1), pairs); c2, b2 = f(make_judge(2.0, 2), pairs)
    assert 0 <= c0 <= 1 and 0 <= b0 <= 1
    assert c2 < c0 - 0.15, f"a position-biased judge must be less consistent ({c2:.2f} vs {c0:.2f})"
    assert b2 > 0.4 and b0 < 0.2, f"first-position bias rate should jump ({b0:.2f} -> {b2:.2f})"

check("swap_consistency", _test_swap_consistency, swap_consistency)

## Task 2: Fit Bradley-Terry strengths

Chatbot Arena estimates model strengths $\xi$ by maximum likelihood under the Bradley-Terry model, $P(i \succ j) = \sigma(\xi_i - \xi_j)$ (with confidence intervals from a sandwich estimator or bootstrap).
Fit $\xi$ by gradient **ascent** on the mean log-likelihood $\frac{1}{N}\sum \log\sigma(\xi_w - \xi_l)$. The gradient adds $(1-\sigma(\xi_w-\xi_l))$ to the winner and subtracts it from the loser. Only differences are identified, so re-centre $\xi$ to mean 0 at the end.

In [ ]:
def fit_bradley_terry(votes, m, steps=2000, lr=2.0):
    xi = np.zeros(m)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_fit_bradley_terry(f):
    xi = f(VOTES, M)
    assert np.isclose(xi.mean(), 0, atol=1e-8), "centre the strengths"
    assert np.allclose(xi, TRUE - TRUE.mean(), atol=0.08), f"should recover true strengths: {np.round(xi, 2)} vs {np.round(TRUE - TRUE.mean(), 2)}"
    perm = np.random.default_rng(5).permutation(len(VOTES))
    assert np.allclose(f(VOTES[perm], M), xi, atol=1e-6), "BT must not depend on vote order"

check("fit_bradley_terry", _test_fit_bradley_terry, fit_bradley_terry)

## Task 3: Online Elo

Elo processes votes **one at a time**: expected score $E_w = 1/(1 + 10^{(R_l - R_w)/400})$, then $R_w \mathrel{+}= K(1 - E_w)$, $R_l \mathrel{-}= K(1 - E_w)$. Start everyone at 1000.
It's cheap and online, but the final ratings depend on the **order** of the votes and on K. The test shuffles the same votes and checks that the ratings move.

In [ ]:
def online_elo(votes, m, k=4.0, init=1000.0):
    R = np.full(m, init)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_online_elo(f):
    R = f(VOTES[:5000], M)
    assert np.argsort(-R).tolist() == list(range(M)), f"ranking should match the true order, got {np.argsort(-R).tolist()}"
    R2 = f(VOTES[:5000][::-1], M)
    assert np.abs(R - R2).max() > 1.0, "Elo should depend on vote order"
    assert np.isclose(f(np.array([[0, 1]]), 2, k=32)[0], 1016.0), "one win between equals moves K/2 = 16 points"

check("online_elo", _test_online_elo, online_elo)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Position-swap consistency
def swap_consistency_solution(judge, pairs):
    cons = first = 0
    for qa, qb in pairs:
        x, y = judge(qa, qb), judge(qb, qa)        # y == 'B' means it again chose the answer that was qa
        cons += (x == 'A') == (y == 'B')
        first += x == 'A' and y == 'A'
    return cons / len(pairs), first / len(pairs)

check("swap_consistency (reference)", _test_swap_consistency, swap_consistency_solution)

In [ ]:
# @title Solution for Task 2: Fit Bradley-Terry strengths
def fit_bradley_terry_solution(votes, m, steps=2000, lr=2.0):
    xi = np.zeros(m); w, l = votes[:, 0], votes[:, 1]
    for _ in range(steps):
        g = 1 - sigmoid(xi[w] - xi[l])
        xi += lr * (np.bincount(w, g, m) - np.bincount(l, g, m)) / len(votes)
    return xi - xi.mean()

check("fit_bradley_terry (reference)", _test_fit_bradley_terry, fit_bradley_terry_solution)

In [ ]:
# @title Solution for Task 3: Online Elo
def online_elo_solution(votes, m, k=4.0, init=1000.0):
    R = np.full(m, init)
    for w, l in votes:
        e = 1 / (1 + 10 ** ((R[l] - R[w]) / 400))
        R[w] += k * (1 - e); R[l] -= k * (1 - e)
    return R

check("online_elo (reference)", _test_online_elo, online_elo_solution)

---
## Experiments
**Order dependence and confidence intervals.** Run Elo on five random orderings of the same votes and look at the spread for each model. Then convert Bradley-Terry strengths to the familiar Elo scale ($400/\ln 10$ per log-odds unit) and add bootstrap 95% intervals, which is what an arena leaderboard should show.

In [ ]:
runs = np.array([online_elo_solution(VOTES[np.random.default_rng(s).permutation(len(VOTES))], M, k=16) for s in range(5)])
print("online Elo (K=16) over 5 orderings of identical votes:")
for i in range(M):
    print(f"  model {i}: {runs[:, i].min():7.1f} .. {runs[:, i].max():7.1f}")
scale = 400 / np.log(10); xi = fit_bradley_terry_solution(VOTES, M) * scale + 1000
boots = np.array([fit_bradley_terry_solution(VOTES[np.random.default_rng(100 + b).integers(0, len(VOTES), len(VOTES))], M, steps=600) for b in range(30)]) * scale + 1000
lo, hi = np.percentile(boots, [2.5, 97.5], axis=0)
print("\nBradley-Terry on the Elo scale, with bootstrap 95% CI:")
for i in range(M):
    print(f"  model {i}: {xi[i]:7.1f}  [{lo[i]:7.1f}, {hi[i]:7.1f}]   true {(TRUE[i] - TRUE.mean()) * scale + 1000:7.1f}")

---
## Takeaways
* LLM judges are cheap and agree with humans about as often as humans agree with each other (MT-Bench: >80%), but they have position, verbosity and self-enhancement biases. Always swap positions and control for length.
* Arena-style leaderboards turn pairwise votes into ratings. Bradley-Terry maximum likelihood is order-independent and gives proper confidence intervals; online Elo is order- and K-dependent.
* Report intervals, not just ranks: neighbouring models often overlap.